# 02 — Transfer learning (ResNet-18, MobileNetV2)

Runs the four transfer-learning experiments on the driver-wise split and saves checkpoints and
results to the shared Drive folder.

In [4]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Mount Drive, get the code, install requirements

In [5]:
from google.colab import drive
drive.mount("/content/drive")

import os
REPO = "/content/driver-distraction-detection"
if not os.path.isdir(REPO):
    !git clone https://github.com/yashashree5/driver-distraction-detection.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!git log --oneline -1
!pip install -q -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/driver-distraction-detection
26f50db (HEAD -> main, origin/main, origin/HEAD) Merge pull request #4 from yashashree5/feat/transfer-learning-colab


## 2. Find the dataset ZIP

In [6]:
import glob
EXPECTED = "/content/drive/MyDrive/DL Final Project/data/statefarm_train.zip"
if os.path.exists(EXPECTED):
    ZIP = EXPECTED
else:
    hits = (glob.glob("/content/drive/MyDrive/**/statefarm_train.zip", recursive=True)
            + glob.glob("/content/drive/Shareddrives/**/statefarm_train.zip", recursive=True))
    print("Search hits:", hits)
    if not hits:
        raise FileNotFoundError(
            "statefarm_train.zip not found. Add a shortcut to the shared 'DL Final Project' folder "
            "in My Drive, then re-run the mount cell with drive.mount('/content/drive', force_remount=True)."
        )
    ZIP = hits[0]
print("Using:", ZIP, f"({os.path.getsize(ZIP) / 1e6:.0f} MB)")

Search hits: ['/content/drive/MyDrive/DL Final Project/DL Final Project/data/statefarm_train.zip']
Using: /content/drive/MyDrive/DL Final Project/DL Final Project/data/statefarm_train.zip (956 MB)


## 3. Unzip onto the Colab disk and verify

In [7]:
if not os.path.exists("data/driver_imgs_list.csv"):
    !unzip -q -n "{ZIP}" -d data
!ls data

import pandas as pd
assert os.path.exists("data/driver_imgs_list.csv"), "data/driver_imgs_list.csv missing - check the ZIP layout with !ls -R data | head"
counts = {f"c{i}": len(os.listdir(f"data/imgs/train/c{i}")) for i in range(10)}
print(counts)
print("images on disk:", sum(counts.values()), "| rows in driver_imgs_list.csv:", len(pd.read_csv("data/driver_imgs_list.csv")))
# README: expect 22,424 images

driver_imgs_list.csv  imgs
{'c0': 2489, 'c1': 2267, 'c2': 2317, 'c3': 2346, 'c4': 2326, 'c5': 2312, 'c6': 2325, 'c7': 2002, 'c8': 1911, 'c9': 2129}
images on disk: 22424 | rows in driver_imgs_list.csv: 22424


## 4. Train

One cell per experiment. Each run saves to `DL Final Project/checkpoints/` and `DL Final Project/results/`.
`time` prints the total wall-clock training time (`real`) at the end.

In [8]:
OUT = "/content/drive/MyDrive/DL Final Project/DL Final Project"
os.makedirs(OUT, exist_ok=True)

### resnet18 — frozen backbone

In [9]:
!time python train.py --model resnet18 --split splits/driverwise.csv --output-dir "{OUT}"

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 128MB/s]
resnet18_frozen_driverwise: 5,130 trainable parameters, 42.65 MB on cuda
epoch  1/15  train loss 1.540 acc 0.530  val loss 1.571 acc 0.493  (169s)
epoch  2/15  train loss 0.971 acc 0.725  val loss 1.456 acc 0.536  (164s)
epoch  3/15  train loss 0.796 acc 0.772  val loss 1.558 acc 0.497  (163s)
epoch  4/15  train loss 0.705 acc 0.792  val loss 1.604 acc 0.502  (177s)
epoch  5/15  train loss 0.653 acc 0.805  val loss 1.549 acc 0.522  (174s)
epoch  6/15  train loss 0.616 acc 0.811  val loss 1.563 acc 0.534  (179s)
epoch  7/15  train loss 0.587 acc 0.823  val loss 1.557 acc 0.537  (176s)
epoch  8/15  train loss 0.565 acc 0.826  val loss 1.510 acc 0.544  (175s)
epoch  9/15  train loss 0.551 acc 0.831  val loss 1.499 acc 0.551  (170s)
epoch 10/15  train loss 0.540 acc 0.834  val loss 1.627 acc 0.509  (169s)
epoch 11/15  t

### resnet18 — fine-tuned

In [10]:
!time python train.py --model resnet18 --split splits/driverwise.csv --no-frozen --output-dir "{OUT}"

resnet18_fine_tuned_driverwise: 11,181,642 trainable parameters, 42.65 MB on cuda
epoch  1/15  train loss 0.210 acc 0.940  val loss 0.278 acc 0.912  (177s)
epoch  2/15  train loss 0.028 acc 0.991  val loss 0.342 acc 0.899  (175s)
epoch  3/15  train loss 0.020 acc 0.994  val loss 0.357 acc 0.908  (174s)
epoch  4/15  train loss 0.007 acc 0.998  val loss 0.270 acc 0.935  (172s)
epoch  5/15  train loss 0.016 acc 0.995  val loss 0.355 acc 0.902  (175s)
epoch  6/15  train loss 0.010 acc 0.997  val loss 0.301 acc 0.924  (174s)
epoch  7/15  train loss 0.006 acc 0.999  val loss 0.267 acc 0.922  (176s)
epoch  8/15  train loss 0.003 acc 0.999  val loss 0.247 acc 0.930  (180s)
epoch  9/15  train loss 0.004 acc 0.999  val loss 0.253 acc 0.937  (183s)
epoch 10/15  train loss 0.001 acc 1.000  val loss 0.235 acc 0.938  (184s)
epoch 11/15  train loss 0.001 acc 1.000  val loss 0.270 acc 0.936  (181s)
epoch 12/15  train loss 0.001 acc 1.000  val loss 0.255 acc 0.938  (181s)
epoch 13/15  train loss 0.000 

### mobilenet_v2 — frozen backbone

In [11]:
!time python train.py --model mobilenet_v2 --split splits/driverwise.csv --output-dir "{OUT}"

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth
100% 13.6M/13.6M [00:00<00:00, 132MB/s]
mobilenet_v2_frozen_driverwise: 12,810 trainable parameters, 8.53 MB on cuda
epoch  1/15  train loss 1.566 acc 0.576  val loss 1.516 acc 0.581  (175s)
epoch  2/15  train loss 0.985 acc 0.754  val loss 1.395 acc 0.553  (169s)
epoch  3/15  train loss 0.790 acc 0.792  val loss 1.261 acc 0.586  (172s)
epoch  4/15  train loss 0.702 acc 0.807  val loss 1.256 acc 0.566  (171s)
epoch  5/15  train loss 0.630 acc 0.822  val loss 1.294 acc 0.535  (170s)
epoch  6/15  train loss 0.595 acc 0.835  val loss 1.230 acc 0.573  (170s)
epoch  7/15  train loss 0.577 acc 0.830  val loss 1.200 acc 0.585  (175s)
epoch  8/15  train loss 0.552 acc 0.840  val loss 1.205 acc 0.588  (174s)
epoch  9/15  train loss 0.530 acc 0.842  val loss 1.228 acc 0.576  (179s)
epoch 10/15  train loss 0.522 acc 0.847  val loss 1.264 acc 0.562  (179s)
ep

### mobilenet_v2 — fine-tuned

In [14]:
!time python train.py --model mobilenet_v2 --split splits/driverwise.csv --no-frozen --output-dir "{OUT}"

mobilenet_v2_fine_tuned_driverwise: 2,236,682 trainable parameters, 8.53 MB on cuda
epoch  1/15  train loss 0.467 acc 0.864  val loss 0.468 acc 0.839  (182s)
epoch  2/15  train loss 0.048 acc 0.985  val loss 0.376 acc 0.880  (185s)
epoch  3/15  train loss 0.022 acc 0.994  val loss 0.493 acc 0.842  (182s)
epoch  4/15  train loss 0.014 acc 0.996  val loss 0.543 acc 0.833  (187s)
epoch  5/15  train loss 0.010 acc 0.997  val loss 0.359 acc 0.899  (186s)
epoch  6/15  train loss 0.007 acc 0.998  val loss 0.368 acc 0.891  (180s)
epoch  7/15  train loss 0.006 acc 0.998  val loss 0.419 acc 0.886  (183s)
epoch  8/15  train loss 0.004 acc 0.999  val loss 0.451 acc 0.873  (189s)
epoch  9/15  train loss 0.003 acc 0.999  val loss 0.376 acc 0.903  (185s)
epoch 10/15  train loss 0.003 acc 0.999  val loss 0.390 acc 0.899  (185s)
epoch 11/15  train loss 0.001 acc 1.000  val loss 0.413 acc 0.891  (182s)
epoch 12/15  train loss 0.001 acc 1.000  val loss 0.446 acc 0.886  (188s)
epoch 13/15  train loss 0.00

## 5. Results

In [13]:
import json
rows = []
for model in ["resnet18", "mobilenet_v2"]:
    for mode in ["frozen", "fine_tuned"]:
        tag = f"{model}_{mode}_driverwise"
        res, ckpt = f"{OUT}/results/{tag}.json", f"{OUT}/checkpoints/{tag}.pt"
        if not os.path.exists(res):
            rows.append({"run": tag, "status": "missing"})
            continue
        r = json.load(open(res))
        rows.append({
            "run": tag,
            "test_acc": round(r["test"]["accuracy"], 4),
            "macro_f1": round(r["test"]["macro_f1"], 4),
            "log_loss": round(r["test"]["loss"], 4),
            "epochs_run": len(r["history"]["val_loss"]),
            "best_epoch": r["history"]["best_epoch"],
            "checkpoint": ckpt if os.path.exists(ckpt) else "missing",
        })
pd.DataFrame(rows)

,run,test_acc,macro_f1,log_loss,epochs_run,best_epoch,checkpoint
0,resnet18_frozen_driverwise,0.4463,0.4331,1.7044,14,9,missing
1,resnet18_fine_tuned_driverwise,0.8838,0.8828,0.5530,15,14,missing
2,mobilenet_v2_frozen_driverwise,0.4221,0.3932,1.7317,15,13,missing
3,mobilenet_v2_fine_tuned_driverwise,0.7702,0.7795,0.9305,14,9,missing
